In [ ]:
import pandas as pd
import csv
from io import StringIO

from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(
    file_name,
    encoding='utf-8',
    engine='python'
)

print("Jumlah baris awal:", len(df))
df.head()


In [ ]:
print("MISSING VALUES SEBELUM PERBAIKAN")
print(df.isnull().sum())


MISSING VALUES SEBELUM PERBAIKAN
nama          0
rating     4143
ulasan     4143
tanggal    4143
sumber        0
dtype: int64


In [ ]:
def parse_broken_row(text):
    try:
        reader = csv.reader(StringIO(text))
        row = next(reader)
        if len(row) >= 4:
            return row[0], row[1], row[2], row[3]
    except:
        pass
    return None, None, None, None


In [ ]:
df_fixed = df.copy()

mask_broken = (
    df_fixed['rating'].isna() &
    df_fixed['nama'].str.contains(',', na=False)
)

print("Jumlah baris rusak terdeteksi:", mask_broken.sum())

for idx in df_fixed[mask_broken].index:
    raw_text = df_fixed.at[idx, 'nama']
    nama, rating, ulasan, tanggal = parse_broken_row(raw_text)

    if rating is not None:
        df_fixed.at[idx, 'nama'] = nama
        df_fixed.at[idx, 'rating'] = rating
        df_fixed.at[idx, 'ulasan'] = ulasan
        df_fixed.at[idx, 'tanggal'] = tanggal


Jumlah baris rusak terdeteksi: 4143


/tmp/ipython-input-2713187785.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_fixed.at[idx, 'rating'] = rating


In [ ]:
df_fixed['rating'] = pd.to_numeric(df_fixed['rating'], errors='coerce')
df_fixed['tanggal'] = pd.to_datetime(df_fixed['tanggal'], errors='coerce')


In [ ]:
print("MISSING VALUES SESUDAH PERBAIKAN")
print(df_fixed.isnull().sum())


MISSING VALUES SESUDAH PERBAIKAN
nama       0
rating     1
ulasan     0
tanggal    1
sumber     0
dtype: int64


In [ ]:
output_file = "ulasan_plnmobile_recovered.csv"

df_fixed.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

files.download(output_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>